In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
filepath = "../data/GlobalWeatherRepository.csv"

In [3]:
df = pd.read_csv(filepath)
df.head()

,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,...,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,...,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


In [4]:
df.isna().sum().sum()

np.int64(0)

In [5]:
drop_cols = [
    "country",
    "location_name",
    "timezone",
    "last_updated_epoch",
    "last_updated",
    "temperature_fahrenheit",
    "wind_mph",
    "wind_direction",
    "pressure_in",
    "precip_in",
    "feels_like_fahrenheit",
    "visibility_miles",
    "gust_mph",
    "sunrise",
    "sunset",
    "moonrise",
    "moonset",
    "moon_phase",
    "moon_illumination"
]

In [6]:
df.drop(columns=drop_cols, inplace=True)
df.dtypes

latitude                        float64
longitude                       float64
temperature_celsius             float64
condition_text                   object
wind_kph                        float64
wind_degree                       int64
pressure_mb                     float64
precip_mm                       float64
humidity                          int64
cloud                             int64
feels_like_celsius              float64
visibility_km                   float64
uv_index                        float64
gust_kph                        float64
air_quality_Carbon_Monoxide     float64
air_quality_Ozone               float64
air_quality_Nitrogen_dioxide    float64
air_quality_Sulphur_dioxide     float64
air_quality_PM2.5               float64
air_quality_PM10                float64
air_quality_us-epa-index          int64
air_quality_gb-defra-index        int64
dtype: object

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, root_mean_squared_error

In [8]:
df_num = df.drop(columns=["condition_text", "feels_like_celsius"])  # drop for temp regression
target_feature = "temperature_celsius"

scaler = StandardScaler()

data_x = df_num.drop(columns=[target_feature]).to_numpy()
data_x = scaler.fit_transform(data_x)
data_y = df_num[target_feature].to_numpy()

x_train, x_test, y_train, y_test = train_test_split(data_x, data_y, test_size=0.25)
x_train.shape, y_test.shape

((123374, 19), (41125,))

In [9]:
# baseline
avg = y_train.mean()
mae = np.mean(np.abs(y_test - avg))
rmse = np.sqrt(np.mean((y_test - avg) ** 2))
f"baseline: MAE: {mae:.2f}, RMSE = {rmse:.2f}, avg value = {y_test.mean():.2f}"

'baseline: MAE: 7.35, RMSE = 9.34, avg value = 21.49'

In [10]:
lin_regr = LinearRegression()
lin_regr.fit(x_train, y_train)
y_pred = lin_regr.predict(x_test)

mae = mean_absolute_error(y_pred, y_test)
f"MAE: {mae:.2f}, avg value = {y_test.mean():.2f}"

# mape isnt stable because temperature can be 0

'MAE: 5.57, avg value = 21.49'

In [11]:
lasso_regr = Lasso(alpha=1.)
lasso_regr.fit(x_train, y_train)
y_pred = lasso_regr.predict(x_test)

mae = mean_absolute_error(y_pred, y_test)
f"MAE: {mae:.2f}, avg value = {y_test.mean():.2f}"

'MAE: 5.86, avg value = 21.49'

In [12]:
tree = DecisionTreeRegressor(criterion="squared_error", max_depth=15,)
tree.fit(x_train, y_train)
y_pred = tree.predict(x_test)

mae = mean_absolute_error(y_pred, y_test)
rmse = root_mean_squared_error(y_pred, y_test)
f"MAE: {mae:.2f}, RMSE = {rmse:.2f}, avg value = {y_test.mean():.2f}"

'MAE: 2.33, RMSE = 3.58, avg value = 21.49'

In [13]:
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(x_train, y_train)
y_pred = knn.predict(x_test)

mae = mean_absolute_error(y_pred, y_test)
rmse = root_mean_squared_error(y_pred, y_test)
f"MAE: {mae:.2f}, RMSE = {rmse:.2f}, avg value = {y_test.mean():.2f}"

'MAE: 2.47, RMSE = 3.79, avg value = 21.49'

In [19]:
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [22]:
param_grid = {
    "max_depth": [3, 5, 7, 10, None],
    "n_estimators": [20, 30, 50, 100],
    "min_samples_split": [2, 5, 10]
}

gs = RandomizedSearchCV(
    estimator=RandomForestRegressor(),
    param_distributions=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_iter=20,
    n_jobs=-1,
    verbose=3
)

gs.fit(x_train, y_train)
y_pred = gs.predict(x_test)

mae = mean_absolute_error(y_pred, y_test)
rmse = root_mean_squared_error(y_pred, y_test)
f"MAE: {mae:.2f}, RMSE = {rmse:.2f}, avg value = {y_test.mean():.2f}"

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END max_depth=5, min_samples_split=2, n_estimators=20;, score=-25.397 total time=  11.8s
[CV 2/5] END max_depth=5, min_samples_split=2, n_estimators=20;, score=-25.465 total time=  13.1s
[CV 1/5] END max_depth=7, min_samples_split=5, n_estimators=20;, score=-19.072 total time=  16.4s
[CV 3/5] END max_depth=7, min_samples_split=5, n_estimators=20;, score=-18.719 total time=  16.7s
[CV 5/5] END max_depth=7, min_samples_split=5, n_estimators=20;, score=-19.198 total time=  16.3s
[CV 2/5] END max_depth=7, min_samples_split=5, n_estimators=20;, score=-19.246 total time=  16.8s
[CV 4/5] END max_depth=7, min_samples_split=5, n_estimators=20;, score=-19.515 total time=  17.2s
[CV 3/5] END max_depth=5, min_samples_split=2, n_estimators=20;, score=-25.256 total time=  11.7s
[CV 4/5] END max_depth=5, min_samples_split=2, n_estimators=20;, score=-25.031 total time=  12.6s
[CV 5/5] END max_depth=5, min_samples_split=2, n_estimat

'MAE: 1.73, RMSE = 2.64, avg value = 21.49'